## Schritt 1: Normalisieren des Excel-Tabellenblatts

Die Ausgangstabelle `03_nicht_normalisierte_Daten.xlsx` enthielt 9 Spalten mit redundanten Werten:

| Problem | Lösung |
|---------|--------|
| `KundeName` — Vor- & Nachname in einem Feld | → `KundeVorname` + `KundeNachname` |
| `KundeAdresse` — Straße, PLZ, Ort in einem Feld | → drei separate Felder |
| `Kategorie` — wiederholt sich bei jedem Produkt | → eigene Tabelle `prodkategorie` |
| `ProduktPreis` — redundant bei jeder Bestellung | → eigene Tabelle `produkte` |

**Ergebnis:** 4 Tabellen nach 3. Normalform — `prodkategorie`, `produkte`, `kunden`, `verkaeufe`  
**Tool:** Excel → 4 CSV-Dateien  
> Die SQL-Tabellenstruktur (CREATE TABLE, Primär- & Fremdschlüssel) befindet sich in `database_setup.sql`.

## Schritt 2: CSV-Daten in PostgreSQL importieren

Die normalisierten Tabellen wurden bereits per SQL in DBeaver erstellt (`database_setup.sql`).  
Hier werden die CSV-Dateien mit pandas eingelesen und in die Datenbank geschrieben.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Umgebungsvariablen aus .env laden
# Passwörter und Zugangsdaten niemals direkt im Code speichern!
load_dotenv()

True

In [2]:
# Datenbankverbindung mit Werten aus .env aufbauen
engine = create_engine(
    f"postgresql+psycopg2://"
    f"{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}"
    f"/{os.environ['DB_NAME']}"
)

In [6]:
# Reihenfolge wichtig: zuerst Kategorien, dann Produkte (wegen Fremdschlüssel)
# encoding='utf-8-sig' entfernt die BOM-Markierung am Anfang der CSV-Dateien
df_kategorie = pd.read_csv("prodkategorie.csv", sep=";", encoding="utf-8-sig")
df_kategorie.to_sql("prodkategorie", engine, if_exists="append", index=False)

df_produkte = pd.read_csv("produkte.csv", sep=";", encoding="utf-8-sig")
df_produkte.to_sql("produkte", engine, if_exists="append", index=False)

# Postleitzahl und Kreditkarte als Text -- führende Nullen dürfen nicht verloren gehen
df_kunden = pd.read_csv(
    "kunden.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype={"KundePostzahl": str, "KundeKreditkarte": str}
)
df_kunden.to_sql("kunden", engine, if_exists="append", index=False)

# Verkäufe zuletzt, da sie auf kunden und produkte verweisen
# parse_dates + dayfirst=True: Datum liegt im Format DD.MM.YYYY vor
df_verkaeufe = pd.read_csv(
    "verkaeufe.csv",
    sep=";",
    encoding="utf-8-sig",
    parse_dates=["Bestelldatum"],
    dayfirst=True
)
df_verkaeufe.to_sql("verkaeufe", engine, if_exists="append", index=False)

100

## Schritt 3 & 4: Rollen, Benutzer und Sicherheits-View

Diese Schritte wurden in **DBeaver** und **pgAdmin** durchgeführt — siehe `database_setup.sql`.

| Schritt | Was | Tool |
|---------|-----|------|
| 3 | Rollen `callcenter_mitarbeiter` und `datenanalyst` erstellt | DBeaver |
| 3 | Benutzer `mitarbeiter` und `schmidt` angelegt (Passwort via `.env`) | DBeaver |
| 4 | View `kunden_view_sicher` — Kreditkarte maskiert, nur letzte 3 Ziffern | DBeaver |
| 4 | `REVOKE` auf `kunden` für PUBLIC — nur View-Zugriff erlaubt | DBeaver |


## Schritt 5: Datenanalyse

Alle Tabellen aus der Datenbank laden und folgende Fragen beantworten:
- Welcher Kunde hat am meisten bezahlt?
- Über welchen Zeitraum laufen die Bestelldaten?
- An welchem Wochentag wird am häufigsten eingekauft?

In [7]:
# Alle Tabellen aus der Datenbank laden
df_verkaeufe = pd.read_sql("SELECT * FROM verkaeufe", engine)
df_kunden    = pd.read_sql("SELECT * FROM kunden", engine)
df_produkte  = pd.read_sql("SELECT * FROM produkte", engine)
df_kategorie = pd.read_sql("SELECT * FROM prodkategorie", engine)

In [8]:
# Verkäufe mit Produkt- und Kundendaten zusammenführen
df_merged = df_verkaeufe.merge(df_produkte, on="ProduktID").merge(df_kunden, on="KundenID")

# Gesamtausgaben pro Bestellung berechnen
df_merged["Gesamtpreis"] = df_merged["Menge"] * df_merged["ProduktPreis"]

# Gesamtausgaben pro Kunde summieren und absteigende sortieren
df_sum = df_merged.groupby(["KundenID", "KundeVorname", "KundeNachname"])["Gesamtpreis"].sum().reset_index()
top_kunden = df_sum.sort_values("Gesamtpreis", ascending=False).head(5)

print("Top 5 Kunden nach Gesamtausgaben:")
print(top_kunden)

Top 5 Kunden nach Gesamtausgaben:
    KundenID KundeVorname KundeNachname  Gesamtpreis
18        19       Linnet       Firmage       121378
9         10       Glenna      Ettritch        53150
0          1         Alla         Mawne        50450
25        26          Ruy       McCarry        37200
23        24        Orton           Tye        33107


In [10]:
# Zeitraum der Bestelldaten ausgeben
# Datum ist bereits als datetime64 gespeichert -- keine Umwandlung nötig
print("Bestellungen von:", df_verkaeufe["Bestelldatum"].min())
print("Bestellungen bis:", df_verkaeufe["Bestelldatum"].max())

Bestellungen von: 2020-10-21
Bestellungen bis: 2023-06-04


In [12]:
# Wochentag aus dem Bestelldatum ableiten und Häufigkeit zählen
# locale wird nicht gesetzt -- funktioniert auf allen Betriebssystemen zuverlässig
df_verkaeufe["Bestelldatum"] = pd.to_datetime(df_verkaeufe["Bestelldatum"])
df_verkaeufe["Wochentag"] = df_verkaeufe["Bestelldatum"].dt.day_name()
wochentage = df_verkaeufe["Wochentag"].value_counts()

print("Bestellungen nach Wochentag:")
print(wochentage)

Bestellungen nach Wochentag:
Wochentag
Wednesday    21
Thursday     17
Tuesday      16
Sunday       14
Saturday     14
Friday       13
Monday        5
Name: count, dtype: int64


## Schritt 6: Datenzugriffsprotokollierung — Trigger

Wurde in **DBeaver** implementiert — siehe `database_setup.sql`.

Ein Trigger auf der Tabelle `kunden` schreibt bei `INSERT`, `UPDATE`, `DELETE` automatisch
einen Eintrag in `zugriffsprotokoll` (Benutzer, Tabelle, Aktion, Zeitstempel).

> ⚠ PostgreSQL unterstützt keine Trigger auf `SELECT` — für vollständiges Lese-Audit wäre `pgaudit` erforderlich.

## Schritt 7: Zugriff mit eingeschränkter Rolle prüfen

Die Verbindung wird jetzt als Benutzer `schmidt` (Rolle: `datenanalyst`) aufgebaut.  
Diese Rolle darf nur lesen -- Schreibversuche müssen einen Fehler liefern.

In [13]:
# Neue Verbindung als Benutzer schmidt aufbauen
# Zugangsdaten ebenfalls aus .env laden
engine_analyst = create_engine(
    f"postgresql+psycopg2://"
    f"{os.environ['DB_USER_ANALYST']}:{os.environ['DB_PASSWORD_ANALYST']}"
    f"@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}"
    f"/{os.environ['DB_NAME']}"
)

# Lesezugriff testen -- soll funktionieren
df = pd.read_sql("SELECT * FROM verkaeufe", engine_analyst)
print(df.head())

   BestellID  Menge Bestelldatum  KundenID  ProduktID
0          1      8   2020-10-21        19         32
1          2      3   2021-01-22         2         33
2          3      5   2021-02-10         8         27
3          4      4   2021-02-11         6         28
4          5      1   2021-03-11         7          9


## Schritt 8: Erweiterte Analyse — Business-Insights

Sechs SQL-Abfragen mit `JOIN`, `GROUP BY`, `HAVING`, `CASE WHEN` und Window Functions.  
Die vollständigen Abfragen sind in `insights.sql` dokumentiert.

### Insight 1: Umsatz pro Produktkategorie

Uhren und Schmuck dominieren trotz weniger Bestellungen — weil Einzelpreise sehr hoch sind.

In [14]:
df_kategorien = pd.read_sql("""
    SELECT
        k."Kategorie",
        SUM(v."Menge" * p."ProduktPreis")          AS Gesamtumsatz,
        COUNT(v."BestellID")                        AS Anzahl_Bestellungen,
        ROUND(AVG(v."Menge" * p."ProduktPreis"), 2) AS Durchschnittlicher_Bestellwert
    FROM verkaeufe v
    JOIN produkte      p ON v."ProduktID"       = p."ProduktID"
    JOIN prodkategorie k ON p."ProdkategorieID" = k."ProdkategorieID"
    GROUP BY k."Kategorie"
    ORDER BY Gesamtumsatz DESC
""", engine_analyst)

print(df_kategorien.to_string(index=False))
# Ergebnis: Uhren (123.680 €) und Schmuck (93.666 €) machen über 50% des Gesamtumsatzes aus

              Kategorie  gesamtumsatz  anzahl_bestellungen  durchschnittlicher_bestellwert
                  Uhren        123680                    4                        30920.00
                Schmuck         93666                   13                         7205.08
                  Möbel         87300                    3                        29100.00
   Computer und Zubehör         20900                    3                         6966.67
       Musikinstrumente         12750                    2                         6375.00
               Haushalt         11530                    6                         1921.67
               Kleidung          6030                    8                          753.75
                   Baby          5906                   15                          393.73
     Sport und Freizeit          3440                    6                          573.33
   Schönheit und Pflege          2945                    3                          981.67

### Insight 2: Monatlicher Umsatz — Saisonalität

`DATE_TRUNC` schneidet das Datum auf Monatsebene ab — ideal um Trends zu erkennen.

In [15]:
df_monate = pd.read_sql("""
    SELECT
        DATE_TRUNC('month', v."Bestelldatum") AS Monat,
        SUM(v."Menge" * p."ProduktPreis")     AS Monatsumsatz,
        COUNT(v."BestellID")                  AS Anzahl_Bestellungen
    FROM verkaeufe v
    JOIN produkte p ON v."ProduktID" = p."ProduktID"
    GROUP BY Monat
    ORDER BY Monatsumsatz DESC
""", engine_analyst)

print(df_monate.to_string(index=False))
# Ergebnis: Oktober 2020 mit 120.000 € -- eine einzige Großbestellung (8x Rolex) treibt den Wert

                    monat  monatsumsatz  anzahl_bestellungen
2020-10-01 00:00:00+02:00        120000                    1
2022-08-01 00:00:00+02:00         71201                    6
2022-07-01 00:00:00+02:00         60721                    9
2021-03-01 00:00:00+01:00         51072                    4
2023-04-01 00:00:00+02:00         16020                    5
2023-05-01 00:00:00+02:00         10485                    6
2022-12-01 00:00:00+01:00          8840                    6
2022-09-01 00:00:00+02:00          8764                    7
2022-10-01 00:00:00+02:00          6266                   10
2023-03-01 00:00:00+01:00          6044                    4
2023-02-01 00:00:00+01:00          5775                    7
2022-06-01 00:00:00+02:00          3236                    2
2023-01-01 00:00:00+01:00          2670                    4
2021-09-01 00:00:00+02:00          2140                    2
2022-11-01 00:00:00+01:00          1916                    6
2021-12-01 00:00:00+01:0

### Insight 3: Kundensegmentierung

`CASE WHEN` klassifiziert Kunden direkt in der Abfrage — ohne zusätzliche Python-Logik.

In [18]:
df_segmente = pd.read_sql("""
    SELECT
        k."KundeVorname",
        k."KundeNachname",
        COUNT(v."BestellID")              AS Anzahl_Bestellungen,
        SUM(v."Menge" * p."ProduktPreis") AS Gesamtumsatz,
        CASE
            WHEN COUNT(v."BestellID") = 1  THEN 'Einmalkäufer'
            WHEN COUNT(v."BestellID") <= 3 THEN 'Gelegentlich'
            ELSE 'Stammkunde'
        END AS Kundensegment
    FROM kunden k
    JOIN verkaeufe v ON k."KundenID"  = v."KundenID"
    JOIN produkte  p ON v."ProduktID" = p."ProduktID"
    GROUP BY k."KundenID", k."KundeVorname", k."KundeNachname"
    ORDER BY Anzahl_Bestellungen DESC
""", engine_analyst)


print(df_segmente['kundensegment'].value_counts().to_string())
print()
print(df_segmente.to_string(index=False))
# Ergebnis: 20% der Kunden (6 von 30) haben nur einmal bestellt

kundensegment
Stammkunde      15
Gelegentlich     9
Einmalkäufer     6

   KundeVorname KundeNachname  anzahl_bestellungen  gesamtumsatz kundensegment
            Edy         Delea                    8          7520    Stammkunde
         Sidnee      Jackalin                    6          8560    Stammkunde
          Orton           Tye                    5         33107    Stammkunde
        Dorella          Eles                    5          2790    Stammkunde
          Heddi       Radford                    5          1559    Stammkunde
         Bobbee      McClaren                    5          1362    Stammkunde
            See        Dorman                    5          8570    Stammkunde
            Kim        Dowsey                    4          1394    Stammkunde
          Neile      Patinkin                    4          4162    Stammkunde
        Krissie       Saunper                    4          1894    Stammkunde
         Linnet       Firmage                    4        1

### Insight 4: Top-Produkt pro Kategorie — Window Function

`RANK() OVER (PARTITION BY ...)` ordnet Produkte innerhalb jeder Kategorie —  
ohne für jede Kategorie eine separate Abfrage zu schreiben.

In [20]:
df_top_produkte = pd.read_sql("""
    SELECT "Kategorie", "ProduktName", "Gesamtumsatz", "Rang"
    FROM (
        SELECT
            k."Kategorie",
            p."ProduktName",
            SUM(v."Menge" * p."ProduktPreis") AS "Gesamtumsatz",
            RANK() OVER (
                PARTITION BY k."Kategorie"
                ORDER BY SUM(v."Menge" * p."ProduktPreis") DESC
            ) AS "Rang"
        FROM verkaeufe v
        JOIN produkte      p ON v."ProduktID"       = p."ProduktID"
        JOIN prodkategorie k ON p."ProdkategorieID" = k."ProdkategorieID"
        GROUP BY k."Kategorie", p."ProduktName"
    ) ranked
    WHERE "Rang" = 1
    ORDER BY "Gesamtumsatz" DESC
""", engine_analyst)

print(df_top_produkte.to_string(index=False))
# Ergebnis: Rolex Submariner (Uhren) und Eames Lounge Chair (Möbel) führen ihre Kategorien deutlich an

              Kategorie                                       ProduktName  Gesamtumsatz  Rang
                  Uhren                        Rolex Submariner Herrenuhr        120000     1
                  Möbel                                Eames Lounge Chair         86800     1
                Schmuck                     Bulgari Serpenti Diamond Ring         84000     1
   Computer und Zubehör                          Apple MacBook Pro Laptop         20900     1
       Musikinstrumente                     Fender Stratocaster E-Gitarre         12750     1
               Haushalt                    Dyson V11 Absolute Staubsauger          9010     1
               Kleidung                                   Zara Trenchcoat          3250     1
   Schönheit und Pflege         Clarisonic Mia 2 Gesichtsreinigungsbürste          2945     1
                   Baby                            BabyBjörn Baby Carrier          2860     1
             Elektronik                     Samsung Galaxy S

### Insight 5: Kunden mit überdurchschnittlichem Bestellwert

`HAVING` filtert nach Aggregaten. Der Subquery berechnet den Gesamtdurchschnitt dynamisch.

In [21]:
df_premium = pd.read_sql("""
    SELECT
        k."KundeVorname",
        k."KundeNachname",
        k."KundeOrt",
        ROUND(AVG(v."Menge" * p."ProduktPreis"), 2) AS Durchschnitt_pro_Bestellung
    FROM kunden k
    JOIN verkaeufe v ON k."KundenID"  = v."KundenID"
    JOIN produkte  p ON v."ProduktID" = p."ProduktID"
    GROUP BY k."KundenID", k."KundeVorname", k."KundeNachname", k."KundeOrt"
    HAVING AVG(v."Menge" * p."ProduktPreis") > (
        SELECT AVG(v2."Menge" * p2."ProduktPreis")
        FROM verkaeufe v2
        JOIN produkte p2 ON v2."ProduktID" = p2."ProduktID"
    )
    ORDER BY Durchschnitt_pro_Bestellung DESC
""", engine_analyst)

print(df_premium.to_string(index=False))
# Diese Premium-Kunden sind besonders wertvoll -- sie kaufen teure Produkte in großen Mengen

KundeVorname KundeNachname           KundeOrt  durchschnitt_pro_bestellung
         Ruy       McCarry             Aachen                     37200.00
      Linnet       Firmage           Dortmund                     30344.50
      Glenna      Ettritch  Frankfurt am Main                     26575.00
        Alla         Mawne         Rheinsberg                     12612.50
       Orton           Tye          Wiesbaden                      6621.40
      Karlen         Holby              Soest                      5186.67
      Mannie     Hendricks            Koblenz                      3975.00


### Insight 6: Umsatzanteil pro Kategorie in Prozent

`SUM() OVER ()` ohne PARTITION berechnet den Gesamtumsatz als Fenster —  
prozentuale Anteile in einer einzigen Abfrage, ohne Subquery.

In [22]:
df_anteile = pd.read_sql("""
    SELECT
        k."Kategorie",
        SUM(v."Menge" * p."ProduktPreis") AS Umsatz,
        ROUND(
            100.0 * SUM(v."Menge" * p."ProduktPreis")
            / SUM(SUM(v."Menge" * p."ProduktPreis")) OVER (),
            1
        ) AS Anteil_Prozent
    FROM verkaeufe v
    JOIN produkte      p ON v."ProduktID"       = p."ProduktID"
    JOIN prodkategorie k ON p."ProdkategorieID" = k."ProdkategorieID"
    GROUP BY k."Kategorie"
    ORDER BY Umsatz DESC
""", engine_analyst)

print(df_anteile.to_string(index=False))
# Ergebnis: Uhren allein = 31% des Gesamtumsatzes. Top-3 Kategorien = über 75%

              Kategorie  umsatz  anteil_prozent
                  Uhren  123680            32.2
                Schmuck   93666            24.4
                  Möbel   87300            22.8
   Computer und Zubehör   20900             5.4
       Musikinstrumente   12750             3.3
               Haushalt   11530             3.0
               Kleidung    6030             1.6
                   Baby    5906             1.5
     Sport und Freizeit    3440             0.9
   Schönheit und Pflege    2945             0.8
Gesundheit und Wellness    2535             0.7
             Bürobedarf    2487             0.6
             Elektronik    2275             0.6
       Filme und Serien    2100             0.5
             Tierbedarf    1905             0.5
      Auto und Motorrad    1800             0.5
                 Bücher    1400             0.4
    Küche und Esszimmer     500             0.1
   Spielzeug und Spiele     400             0.1


In [23]:
# Beide Verbindungen sauber schließen
engine.dispose()
engine_analyst.dispose()